In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sqlalchemy import create_engine

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

In [2]:
DB_USER = "postgres"
DB_PASSWORD = "postgres"
DB_HOST = "localhost"
DB_PORT = "5432"
DB_NAME = "customer_revenue_platform"

engine = create_engine(
    f"postgresql+psycopg2://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
)

print("Connected Successfully")

Connected Successfully


In [3]:
query = """
SELECT *
FROM customer_features
"""

customer_features = pd.read_sql(
    query,
    engine
)

In [4]:
customer_features.shape
customer_features.head()
customer_features.info()

<class 'pandas.DataFrame'>
RangeIndex: 96096 entries, 0 to 96095
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   customer_unique_id      96096 non-null  str           
 1   total_orders            96096 non-null  int64         
 2   total_revenue           96096 non-null  float64       
 3   avg_order_value         96096 non-null  float64       
 4   avg_review_score        96096 non-null  float64       
 5   total_freight           96096 non-null  float64       
 6   avg_installments        96096 non-null  float64       
 7   first_purchase          96096 non-null  datetime64[us]
 8   last_purchase           96096 non-null  datetime64[us]
 9   customer_lifetime_days  96096 non-null  int64         
 10  recency_days            96096 non-null  int64         
 11  repeat_customer         96096 non-null  int64         
 12  customer_tenure_months  96096 non-null  float64       
 1

In [5]:
customer_features.columns.tolist()

['customer_unique_id',
 'total_orders',
 'total_revenue',
 'avg_order_value',
 'avg_review_score',
 'total_freight',
 'avg_installments',
 'first_purchase',
 'last_purchase',
 'customer_lifetime_days',
 'recency_days',
 'repeat_customer',
 'customer_tenure_months',
 'revenue_per_day',
 'high_value_customer',
 'freight_percentage',
 'review_category',
 'recency_group',
 'R_score',
 'F_score',
 'M_score',
 'RFM_score',
 'customer_value_score']

In [6]:
# target variable
y = customer_features["repeat_customer"]

In [7]:
# checking for repeat customers
customer_features["repeat_customer"].value_counts()

repeat_customer
0    93099
1     2997
Name: count, dtype: int64

In [8]:
# checking for repeat customers in percentage
customer_features["repeat_customer"].value_counts(normalize=True) * 100

repeat_customer
0    96.881244
1     3.118756
Name: proportion, dtype: float64

In [9]:
# checking for missng values
customer_features.isnull().sum()

customer_unique_id          0
total_orders                0
total_revenue               0
avg_order_value             0
avg_review_score            0
total_freight               0
avg_installments            0
first_purchase              0
last_purchase               0
customer_lifetime_days      0
recency_days                0
repeat_customer             0
customer_tenure_months      0
revenue_per_day             0
high_value_customer         0
freight_percentage          0
review_category           716
recency_group               2
R_score                     0
F_score                     0
M_score                     0
RFM_score                   0
customer_value_score        0
dtype: int64

In [10]:
# checking for repeat customers
customer_features["repeat_customer"].value_counts()

customer_features["repeat_customer"].value_counts(normalize=True) * 100

customer_features.isnull().sum()

customer_unique_id          0
total_orders                0
total_revenue               0
avg_order_value             0
avg_review_score            0
total_freight               0
avg_installments            0
first_purchase              0
last_purchase               0
customer_lifetime_days      0
recency_days                0
repeat_customer             0
customer_tenure_months      0
revenue_per_day             0
high_value_customer         0
freight_percentage          0
review_category           716
recency_group               2
R_score                     0
F_score                     0
M_score                     0
RFM_score                   0
customer_value_score        0
dtype: int64

In [11]:
# 
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

In [12]:
customer_features["review_category"] = (
    customer_features["review_category"]
    .fillna("Unknown")
)

customer_features["recency_group"] = (
    customer_features["recency_group"]
    .fillna("Unknown")
)

In [13]:
customer_features.isnull().sum().sum()

np.int64(0)

In [14]:
y = customer_features["repeat_customer"]

In [15]:
leakage_columns = [
    "total_orders",
    "F_score",
    "RFM_score",
    "customer_value_score"
]

In [16]:
X = customer_features.drop(
    columns=[
        "customer_unique_id",
        "repeat_customer",
        "first_purchase",
        "last_purchase"
    ] + leakage_columns
)

In [17]:
X = pd.get_dummies(
    X,
    drop_first=True
)

In [18]:
import numpy as np

X.replace(
    [np.inf, -np.inf],
    0,
    inplace=True
)

,total_revenue,avg_order_value,avg_review_score,total_freight,avg_installments,customer_lifetime_days,recency_days,customer_tenure_months,revenue_per_day,high_value_customer,...,recency_group_Very Recent,recency_group_Warm,R_score_2,R_score_3,R_score_4,R_score_5,M_score_2,M_score_3,M_score_4,M_score_5
0,141.90,141.90,5.0,12.00,8.0,0,160,0.0,141.90,0,...,False,True,False,False,True,False,False,False,True,False
1,27.19,27.19,4.0,8.29,1.0,0,163,0.0,27.19,0,...,False,True,False,False,True,False,False,False,False,False
2,86.22,86.22,3.0,17.22,8.0,0,585,0.0,86.22,0,...,False,False,False,False,False,False,True,False,False,False
3,43.62,43.62,4.0,17.63,4.0,0,369,0.0,43.62,0,...,False,False,True,False,False,False,False,False,False,False
4,196.89,196.89,5.0,16.89,6.0,0,336,0.0,196.89,0,...,False,False,True,False,False,False,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96091,4134.84,2067.42,5.0,497.42,10.0,0,495,0.0,4134.84,1,...,False,False,False,False,False,False,False,False,False,True
96092,84.58,84.58,4.0,19.69,1.0,0,310,0.0,84.58,0,...,False,False,False,True,False,False,True,False,False,False
96093,112.46,112.46,5.0,22.56,1.0,0,617,0.0,112.46,0,...,False,False,False,False,False,False,False,True,False,False
96094,133.69,133.69,5.0,18.69,5.0,0,168,0.0,133.69,0,...,False,True,False,False,True,False,False,True,False,False


In [19]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [20]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [21]:
from sklearn.linear_model import LogisticRegression

lr_model = LogisticRegression(
    max_iter=3000,
    class_weight="balanced"
)

lr_model.fit(
    X_train,
    y_train
)

,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",'balanced'
,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",3000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Defaul

In [22]:
from sklearn.metrics import classification_report

y_pred = lr_model.predict(X_test)

print(
    classification_report(
        y_test,
        y_pred
    )
)

              precision    recall  f1-score   support

           0       1.00      0.95      0.97     18621
           1       0.39      0.91      0.54       599

    accuracy                           0.95     19220
   macro avg       0.69      0.93      0.76     19220
weighted avg       0.98      0.95      0.96     19220



In [23]:
classification_report(y_test, y_pred)

'              precision    recall  f1-score   support\n\n           0       1.00      0.95      0.97     18621\n           1       0.39      0.91      0.54       599\n\n    accuracy                           0.95     19220\n   macro avg       0.69      0.93      0.76     19220\nweighted avg       0.98      0.95      0.96     19220\n'

In [24]:
cm = confusion_matrix(y_test, y_pred)

print(cm)

[[17762   859]
 [   53   546]]


In [25]:
print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

Accuracy : 0.9525494276795006
Precision: 0.38861209964412813
Recall   : 0.9115191986644408
F1 Score : 0.5449101796407185


In [26]:
train_pred = lr_model.predict(X_train)

print(
    "Train Accuracy:",
    accuracy_score(y_train, train_pred)
)

print(
    "Test Accuracy:",
    accuracy_score(y_test, y_pred)
)

Train Accuracy: 0.9536916592954888
Test Accuracy: 0.9525494276795006


In [27]:
importance = pd.DataFrame({
    "feature": X.columns,
    "coefficient": lr_model.coef_[0]
})

importance.sort_values(
    by="coefficient",
    ascending=False
).head(20)

,feature,coefficient
0,total_revenue,11.018322
5,customer_lifetime_days,9.921505
7,customer_tenure_months,9.867186
26,M_score_5,1.801382
25,M_score_4,1.508232
24,M_score_3,1.030020
3,total_freight,0.861839
23,M_score_2,0.616758
6,recency_days,0.347984
11,review_category_Excellent,0.260834


In [28]:
prob = lr_model.predict_proba(X_test)

prob[:5]

array([[0.95802008, 0.04197992],
       [0.97353519, 0.02646481],
       [0.93512637, 0.06487363],
       [0.98823978, 0.01176022],
       [0.93512506, 0.06487494]])

In [29]:
from sklearn.metrics import roc_auc_score

prob = lr_model.predict_proba(X_test)[:,1]

roc_auc_score(
    y_test,
    prob
)

0.9811379867220479

In [30]:
model_results = pd.DataFrame({
    "Model": ["LOGISTIC REGRESSION"],
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test,y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1": f1_score(y_test, y_pred),
    "ROC_AUC": roc_auc_score(y_test, prob),
})

model_results

,Model,Accuracy,Precision,Recall,F1,ROC_AUC
0,LOGISTIC REGRESSION,0.952549,0.388612,0.911519,0.54491,0.981138


In [31]:
# STORING ACCURACY, PRECISION, RECALL, F1 SCORE AND ROC AUC OF LOGISTIC REGRESSION IN POSTGRESQL DATABASE
from datetime import datetime

lr_results = pd.DataFrame({
    "model_name": ["Logistic Regression"],
    "accuracy": [accuracy_score(y_test, y_pred)],
    "precision": [precision_score(y_test, y_pred)],
    "recall": [recall_score(y_test, y_pred)],
    "f1_score": [f1_score(y_test, y_pred)],
    "roc_auc": [roc_auc_score(y_test, prob)],
    "run_date": [datetime.now()]
})

from sqlalchemy import text

model_name = "Logistic Regression"

with engine.begin() as conn:
    conn.execute(
        text("DELETE FROM model_metrics WHERE model_name = :model"),
        {"model": model_name}
    )

lr_results.to_sql(
    "model_metrics",
    con=engine,
    if_exists="append",
    index=False
)

1

In [32]:
import joblib
from pathlib import Path

BASE_DIR = Path.cwd()

MODEL_DIR = BASE_DIR / "Trained_Models" / "purchase_prediction"

MODEL_DIR.mkdir(
    parents=True,
    exist_ok=True
)

joblib.dump(
    lr_model,
    MODEL_DIR / "logistic_regression.pkl"
)

['C:\\Users\\vansh\\OneDrive\\Desktop\\sureTrust\\Trained_Models\\purchase_prediction\\logistic_regression.pkl']